# 02 · Feature Engineering
Demonstrates how raw NHANES variables are transformed into model features.
Mirrors the logic in `src/ml/data/diabetes_preprocessor.py`.

**Topics:**
1. Variable renaming and type conversion
2. Derived features (age groups, BMI categories, smoking status, activity level)
3. Missing value analysis and imputation strategy
4. Feature encoding (binary, ordinal, one-hot)
5. Final feature matrix shape and class balance

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.ml.data.diabetes_preprocessor import DiabetesPreprocessor

sns.set_theme(style='darkgrid', palette='muted')
print('Imports OK.')

## 1 · Run the preprocessor pipeline

In [ ]:
DATA_DIR = '../data/raw/2017_2018'

prep = DiabetesPreprocessor()
X, y = prep.load_and_preprocess(DATA_DIR)

print(f'Feature matrix:  {X.shape}')
print(f'Target vector:   {y.shape}  (positive rate: {y.mean():.1%})')
X.head(3)

## 2 · Feature inventory

In [ ]:
feature_groups = {
    'Demographic':    ['age','gender','education','income_ratio'],
    'Anthropometric': [c for c in X.columns if c in ('bmi','waist_circumference','weight','height')],
    'Lab':            [c for c in X.columns if c in ('hba1c','fasting_glucose','total_cholesterol','hdl_cholesterol')],
    'Lifestyle':      [c for c in X.columns if c in ('sedentary_minutes','vigorous_work','moderate_work','vigorous_rec','moderate_rec')],
    'Medical':        [c for c in X.columns if c in ('family_diabetes','smoked_100','current_smoker')],
    'One-hot':        [c for c in X.columns if any(c.startswith(p) for p in ('age_group','bmi_category','smoking_status','activity_level','race_'))],
}

for grp, cols in feature_groups.items():
    print(f'{grp:15} ({len(cols):2} features): {cols}')

print(f'\nTotal: {X.shape[1]} features')

## 3 · Derived feature distributions

In [ ]:
# Age groups and BMI categories
age_group_cols = [c for c in X.columns if c.startswith('age_group')]
bmi_cols       = [c for c in X.columns if c.startswith('bmi_category')]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

if age_group_cols:
    ag_counts = X[age_group_cols].sum().sort_index()
    ag_counts.index = [c.replace('age_group_','') for c in ag_counts.index]
    ag_counts.plot(kind='bar', ax=axes[0], color='steelblue')
    axes[0].set_title('Age group distribution (one-hot dummies)')
    axes[0].set_xlabel('')
    axes[0].tick_params(axis='x', rotation=0)

if bmi_cols:
    bmi_counts = X[bmi_cols].sum().sort_values(ascending=False)
    bmi_counts.index = [c.replace('bmi_category_','') for c in bmi_counts.index]
    bmi_counts.plot(kind='bar', ax=axes[1], color='coral')
    axes[1].set_title('BMI category distribution (one-hot dummies; ref=Normal)')
    axes[1].set_xlabel('')
    axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

## 4 · Missing value imputation: before vs after

In [ ]:
# The preprocessor fills NaN with feature medians
print('Missing values after preprocessing:')
miss = X.isnull().sum()
print(miss[miss > 0] if miss.sum() > 0 else '  None — all filled ✓')

## 5 · Feature correlation with diabetes target

In [ ]:
corr_with_target = X.corrwith(y.astype(float)).abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
corr_with_target.head(20).plot(kind='barh', color='steelblue', ax=ax)
ax.invert_yaxis()
ax.set_xlabel('|Pearson r| with diabetes label')
ax.set_title('Top 20 features by correlation with diabetes target')
plt.tight_layout()
plt.show()

## 6 · Class balance

In [ ]:
from collections import Counter

counts = Counter(y)
print(f'Negative (no diabetes): {counts[0]:,}  ({counts[0]/len(y):.1%})')
print(f'Positive (diabetes):    {counts[1]:,}  ({counts[1]/len(y):.1%})')
print(f'Class ratio (neg/pos):  {counts[0]/counts[1]:.1f}x')
print()
print('No explicit class weighting is used in training.')
print('XGBoost handles mild imbalance well at this ratio.')

## 7 · Key decisions

| Feature | Engineering decision | Rationale |
|---------|----------------------|-----------|
| **Age** | Continuous + one-hot age group (ref=18-35) | Captures both linear trend and threshold effects |
| **BMI** | Continuous + one-hot BMI category (ref=Normal) | WHO obesity classes matter clinically |
| **Smoking** | 3-level (Never/Former/Current; ref=Current) | Former smokers have different risk to never smokers |
| **Activity** | 4-level (High/Moderate/Low/Sedentary) | Collapses 5 PAQ questions |
| **Gender** | Binary (0=Male, 1=Female) | NHANES has two gender categories |
| **HbA1c / glucose** | Kept continuous; also used in diabetes indicator for CVD/HTN | Direct biochemical markers |
| **BP** | **Excluded from HTN model** — target is defined from BP readings | Prevents trivial circularity |
| **Missing values** | Median imputation per column | Preserves sample size; lab values are MCAR/MAR |

➡️ Continue to `03_model_training.ipynb`